In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

DATA_PATH = Path("../data/raw/resistance_sweep_dataset.csv")
CLEANED_DIR = Path("../data/cleaned")
PROCESSED_DIR = Path("../data/processed")
FIG_DIR = Path("../figures/resistance_sweep")
SOURCE_LOADING_DIR = Path("../figures/source_loading")

CLEANED_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)
SOURCE_LOADING_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.figsize": (10, 6),
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 13,
    "legend.fontsize": 10,
})

In [ ]:
df = pd.read_csv(DATA_PATH)

df.head()

In [ ]:
df.info()

In [ ]:
df = df.copy()

required_cols = [
    "Resistance_Ohm",
    "Frequency_Hz",
    "Vs_Vpp",
    "VR_Vpp",
    "Gain"
]

missing = [col for col in required_cols if col not in df.columns]
if missing:
    raise ValueError(f"Missing columns: {missing}")

for col in required_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

df = df.dropna(subset=required_cols)
df = df.sort_values(["Resistance_Ohm", "Frequency_Hz"]).reset_index(drop=True)

df["Gain_Recomputed"] = df["VR_Vpp"] / df["Vs_Vpp"]
df["Gain_Error"] = df["Gain"] - df["Gain_Recomputed"]

df.to_csv(CLEANED_DIR / "resistance_sweep_cleaned.csv", index=False)

df.head()

In [ ]:
gain_error_threshold = 0.01

suspect_gain = df[np.abs(df["Gain_Error"]) > gain_error_threshold]

suspect_gain

In [ ]:
def flag_local_outliers(group, column="Gain", threshold=0.12):
    group = group.sort_values("Frequency_Hz").copy()
    rolling_median = group[column].rolling(
        window=3,
        center=True,
        min_periods=1
    ).median()
    group["Local_Deviation"] = np.abs(group[column] - rolling_median)
    group["Suspect_Outlier"] = group["Local_Deviation"] > threshold
    return group

df_flagged = (
    df.groupby("Resistance_Ohm", group_keys=False)
      .apply(flag_local_outliers)
      .reset_index(drop=True)
)

df_flagged[df_flagged["Suspect_Outlier"]]

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for R, group in df_flagged.groupby("Resistance_Ohm"):
    group = group.sort_values("Frequency_Hz")

    ax.semilogx(
        group["Frequency_Hz"],
        group["Gain"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=f"{R:.0f} Ω"
    )

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Gain |VR/Vs|")
ax.set_title("Frequency Response Under Varying Resistive Damping")
ax.grid(True, which="both", alpha=0.35)
ax.legend(title="Measured R")

fig.savefig(FIG_DIR / "resistance_sweep_gain_curves.png", dpi=300, bbox_inches="tight")
fig.savefig(FIG_DIR / "resistance_sweep_gain_curves.svg", bbox_inches="tight")

plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(11, 6))

for R, group in df_flagged.groupby("Resistance_Ohm"):
    group = group.sort_values("Frequency_Hz")

    ax.semilogx(
        group["Frequency_Hz"],
        group["Vs_Vpp"],
        marker="o",
        linewidth=2,
        markersize=4,
        label=f"{R:.0f} Ω"
    )

ax.set_xlabel("Frequency (Hz)")
ax.set_ylabel("Measured Source Voltage Vs (Vpp)")
ax.set_title("Source Loading Across Resistance Sweep")
ax.grid(True, which="both", alpha=0.35)
ax.legend(title="Measured R")

fig.savefig(SOURCE_LOADING_DIR / "source_loading_resistance_sweep.png", dpi=300, bbox_inches="tight")
fig.savefig(SOURCE_LOADING_DIR / "source_loading_resistance_sweep.svg", bbox_inches="tight")

plt.show()

In [ ]:
summary_rows = []

for R, group in df_flagged.groupby("Resistance_Ohm"):
    group = group.sort_values("Frequency_Hz")

    f = group["Frequency_Hz"].values
    g = group["Gain"].values
    vs = group["Vs_Vpp"].values

    peak_idx = np.argmax(g)
    peak_gain = g[peak_idx]
    peak_freq = f[peak_idx]

    half_power_gain = peak_gain / np.sqrt(2)
    above = g >= half_power_gain

    if np.sum(above) >= 2:
        f_low = f[above][0]
        f_high = f[above][-1]
        bandwidth = f_high - f_low
        q_approx = peak_freq / bandwidth if bandwidth > 0 else np.nan
    else:
        f_low = np.nan
        f_high = np.nan
        bandwidth = np.nan
        q_approx = np.nan

    vs_baseline = vs[0]
    vs_min = np.min(vs)
    vs_drop = vs_baseline - vs_min
    vs_drop_percent = 100 * vs_drop / vs_baseline

    summary_rows.append({
        "Resistance_Ohm": R,
        "Peak_Gain": peak_gain,
        "Peak_Frequency_Hz": peak_freq,
        "Half_Power_Gain": half_power_gain,
        "f_low_Hz": f_low,
        "f_high_Hz": f_high,
        "Bandwidth_Hz": bandwidth,
        "Q_Approx": q_approx,
        "Vs_Baseline_Vpp": vs_baseline,
        "Vs_Min_Vpp": vs_min,
        "Vs_Drop_Vpp": vs_drop,
        "Vs_Drop_Percent": vs_drop_percent,
    })

summary = pd.DataFrame(summary_rows).sort_values("Resistance_Ohm")
summary.to_csv(PROCESSED_DIR / "resistance_sweep_summary.csv", index=False)

summary

In [ ]:
def save_line_plot(x, y, xlabel, ylabel, title, filename):
    fig, ax = plt.subplots(figsize=(8, 5))

    ax.plot(x, y, marker="o", linewidth=2)

    ax.set_xlabel(xlabel)
    ax.set_ylabel(ylabel)
    ax.set_title(title)
    ax.grid(True, alpha=0.35)

    fig.savefig(FIG_DIR / f"{filename}.png", dpi=300, bbox_inches="tight")
    fig.savefig(FIG_DIR / f"{filename}.svg", bbox_inches="tight")

    plt.show()

In [ ]:
save_line_plot(
    summary["Resistance_Ohm"],
    summary["Peak_Gain"],
    "Measured Resistance R (Ω)",
    "Peak Gain",
    "Peak Gain vs Resistance",
    "peak_gain_vs_resistance"
)

save_line_plot(
    summary["Resistance_Ohm"],
    summary["Peak_Frequency_Hz"],
    "Measured Resistance R (Ω)",
    "Approximate Resonance Frequency (Hz)",
    "Resonance Frequency vs Resistance",
    "resonance_frequency_vs_resistance"
)

save_line_plot(
    summary["Resistance_Ohm"],
    summary["Bandwidth_Hz"],
    "Measured Resistance R (Ω)",
    "Approximate Bandwidth (Hz)",
    "Bandwidth vs Resistance",
    "bandwidth_vs_resistance"
)

save_line_plot(
    summary["Resistance_Ohm"],
    summary["Q_Approx"],
    "Measured Resistance R (Ω)",
    "Approximate Q Factor",
    "Approximate Q Factor vs Resistance",
    "q_factor_vs_resistance"
)

save_line_plot(
    summary["Resistance_Ohm"],
    summary["Vs_Drop_Percent"],
    "Measured Resistance R (Ω)",
    "Source Voltage Drop (%)",
    "Source Loading vs Resistance",
    "source_voltage_drop_vs_resistance"
)